# Milestone 1 - Data Analysis and Exploration

This notebook covers the data analysis component of Milestone 1 for the Product Question Answering System project. We load and explore the Amazon product dataset, examine distributions, identify patterns, and discuss limitations that will inform our modeling approach in later milestones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Loading the Data

In [ ]:
products = pd.read_csv('amazon_products_cleaned.csv')
categories = pd.read_csv('amazon_categories.csv')

print(f'Products dataset shape: {products.shape}')
print(f'Categories dataset shape: {categories.shape}')

In [ ]:
products.head(10)

In [ ]:
categories.head(10)

In [ ]:
# merge the two datasets on category_id
df = products.merge(categories, left_on='category_id', right_on='id', how='left')
df.drop(columns=['id'], inplace=True)
print(f'Merged dataset shape: {df.shape}')
df.head()

## 2. Basic Dataset Statistics

In [ ]:
print('Column types:')
print(df.dtypes)
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print(f'Total duplicate ASINs: {df["asin"].duplicated().sum()}')

In [ ]:
df.describe()

In [ ]:
print(f'Number of unique categories: {df["category_name"].nunique()}')
print(f'Number of unique products (by ASIN): {df["asin"].nunique()}')
print(f'Price range: ${df["price"].min():.2f} - ${df["price"].max():.2f}')
print(f'Average rating (excluding 0): {df[df["stars"] > 0]["stars"].mean():.2f}')
print(f'Products with no rating (stars=0): {(df["stars"] == 0).sum()}')
print(f'Products marked as Best Seller: {df["isBestSeller"].sum()}')

## 3. Category Distribution Analysis

In [ ]:
cat_counts = df['category_name'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# top 20 categories
cat_counts.head(20).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 20 Categories by Product Count')
axes[0].set_xlabel('Number of Products')
axes[0].invert_yaxis()

# bottom 20 categories
cat_counts.tail(20).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Bottom 20 Categories by Product Count')
axes[1].set_xlabel('Number of Products')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'Category with most products: {cat_counts.index[0]} ({cat_counts.iloc[0]})')
print(f'Category with fewest products: {cat_counts.index[-1]} ({cat_counts.iloc[-1]})')
print(f'Median products per category: {cat_counts.median():.0f}')

In [ ]:
# distribution of category sizes (histogram)
plt.figure(figsize=(10, 5))
plt.hist(cat_counts.values, bins=50, color='teal', edgecolor='black', alpha=0.7)
plt.xlabel('Number of Products in Category')
plt.ylabel('Number of Categories')
plt.title('Distribution of Category Sizes')
plt.axvline(cat_counts.median(), color='red', linestyle='--', label=f'Median = {cat_counts.median():.0f}')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Price Analysis

In [ ]:
# filter out zero-price products for meaningful analysis
df_priced = df[df['price'] > 0].copy()
print(f'Products with price > 0: {len(df_priced)} / {len(df)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# price distribution (capped at 99th percentile for visibility)
cap = df_priced['price'].quantile(0.99)
axes[0].hist(df_priced[df_priced['price'] <= cap]['price'], bins=80, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Price Distribution (up to 99th percentile)')

# log-scale price distribution
axes[1].hist(np.log1p(df_priced['price']), bins=80, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('log(1 + Price)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log-Transformed Price Distribution')

plt.tight_layout()
plt.show()

print(f'Mean price: ${df_priced["price"].mean():.2f}')
print(f'Median price: ${df_priced["price"].median():.2f}')
print(f'99th percentile price: ${cap:.2f}')

In [ ]:
# average price by top 15 categories
top_cats = cat_counts.head(15).index.tolist()
avg_price_by_cat = df_priced[df_priced['category_name'].isin(top_cats)].groupby('category_name')['price'].mean().sort_values(ascending=True)

plt.figure(figsize=(12, 6))
avg_price_by_cat.plot(kind='barh', color='teal')
plt.xlabel('Average Price ($)')
plt.title('Average Price for Top 15 Categories')
plt.tight_layout()
plt.show()

## 5. Ratings and Reviews Analysis

In [ ]:
df_rated = df[df['stars'] > 0].copy()
print(f'Products with ratings: {len(df_rated)} / {len(df)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# star rating distribution
axes[0].hist(df_rated['stars'], bins=50, color='gold', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Star Ratings')
axes[0].axvline(df_rated['stars'].mean(), color='red', linestyle='--', label=f'Mean = {df_rated["stars"].mean():.2f}')
axes[0].legend()

# review count distribution (log scale)
df_reviewed = df[df['reviews'] > 0]
axes[1].hist(np.log1p(df_reviewed['reviews']), bins=60, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('log(1 + Review Count)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Review Counts (log scale)')

plt.tight_layout()
plt.show()

print(f'Products with at least 1 review: {len(df_reviewed)} ({100*len(df_reviewed)/len(df):.1f}%)')
print(f'Max reviews on a single product: {df["reviews"].max()}')

In [ ]:
# correlation between stars and reviews (for products that have both)
df_both = df[(df['stars'] > 0) & (df['reviews'] > 0)].copy()

plt.figure(figsize=(8, 6))
plt.scatter(df_both['stars'], np.log1p(df_both['reviews']), alpha=0.1, s=5, color='steelblue')
plt.xlabel('Star Rating')
plt.ylabel('log(1 + Reviews)')
plt.title('Star Rating vs Review Count')
plt.tight_layout()
plt.show()

print(f'Pearson correlation between stars and log(reviews): {np.corrcoef(df_both["stars"], np.log1p(df_both["reviews"]))[0,1]:.3f}')

## 6. Popularity Analysis (boughtInLastMonth)

In [ ]:
popular = df[df['boughtInLastMonth'] > 0].copy()
print(f'Products bought at least once last month: {len(popular)} ({100*len(popular)/len(df):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(np.log1p(popular['boughtInLastMonth']), bins=50, color='seagreen', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('log(1 + Bought in Last Month)')
axes[0].set_ylabel('Count')
axes[0].set_title('Purchase Volume Distribution (log scale)')

# best sellers vs non-best sellers
bs_counts = df['isBestSeller'].value_counts()
axes[1].bar(['Not Best Seller', 'Best Seller'], bs_counts.values, color=['lightcoral', 'seagreen'])
axes[1].set_ylabel('Count')
axes[1].set_title('Best Seller Distribution')
for i, v in enumerate(bs_counts.values):
    axes[1].text(i, v + 200, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# top 10 most bought products
top_bought = df.nlargest(10, 'boughtInLastMonth')[['title', 'category_name', 'price', 'stars', 'boughtInLastMonth']]
top_bought

## 7. Text Analysis on Product Titles

In [ ]:
# title length analysis
df['title_length'] = df['title'].astype(str).apply(len)
df['title_word_count'] = df['title'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['title_length'], bins=80, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Title Length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Title Lengths')
axes[0].axvline(df['title_length'].mean(), color='red', linestyle='--', label=f'Mean = {df["title_length"].mean():.0f}')
axes[0].legend()

axes[1].hist(df['title_word_count'], bins=60, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Title Word Count')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Title Word Counts')
axes[1].axvline(df['title_word_count'].mean(), color='red', linestyle='--', label=f'Mean = {df["title_word_count"].mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Average title length: {df["title_length"].mean():.1f} characters')
print(f'Average title word count: {df["title_word_count"].mean():.1f} words')
print(f'Max title length: {df["title_length"].max()} characters')

In [ ]:
# most common words in product titles (excluding common stop words)
stop_words = {'for', 'and', 'the', 'with', 'a', 'of', 'to', 'in', 'on', 'is', 'it', 'by',
              'an', 'or', 'at', 'as', 'be', 'from', 'that', 'this', 'are', 'was', 'not', '-', '&', '|'}

all_words = []
for title in df['title'].dropna():
    words = re.findall(r'[a-zA-Z]+', str(title).lower())
    all_words.extend([w for w in words if w not in stop_words and len(w) > 2])

word_counts = Counter(all_words)
top_words = word_counts.most_common(30)

plt.figure(figsize=(14, 6))
words, counts = zip(*top_words)
plt.bar(words, counts, color='teal', edgecolor='black', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.title('Top 30 Most Frequent Words in Product Titles')
plt.tight_layout()
plt.show()

## 8. Discount Analysis

In [ ]:
# products with a list price (i.e., discounted products)
df_discounted = df[(df['listPrice'] > 0) & (df['price'] > 0)].copy()
df_discounted['discount_pct'] = ((df_discounted['listPrice'] - df_discounted['price']) / df_discounted['listPrice']) * 100

# keep only reasonable discounts (0-100%)
df_discounted = df_discounted[(df_discounted['discount_pct'] >= 0) & (df_discounted['discount_pct'] <= 100)]

print(f'Products with valid discount info: {len(df_discounted)} ({100*len(df_discounted)/len(df):.1f}%)')
print(f'Average discount: {df_discounted["discount_pct"].mean():.1f}%')
print(f'Median discount: {df_discounted["discount_pct"].median():.1f}%')

plt.figure(figsize=(10, 5))
plt.hist(df_discounted['discount_pct'], bins=50, color='crimson', edgecolor='black', alpha=0.7)
plt.xlabel('Discount Percentage')
plt.ylabel('Count')
plt.title('Distribution of Discount Percentages')
plt.axvline(df_discounted['discount_pct'].mean(), color='blue', linestyle='--', label=f'Mean = {df_discounted["discount_pct"].mean():.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Correlation Heatmap

In [ ]:
numeric_cols = ['stars', 'reviews', 'price', 'listPrice', 'isBestSeller', 'boughtInLastMonth', 'title_length', 'title_word_count']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

## 10. QA-Relevant Data Preparation Analysis

For the Question Answering system, we need to construct context passages from the product metadata. Below we examine how informative the textual content is and what kind of questions could be answered.

In [ ]:
# constructing a sample context passage for QA
def build_context(row):
    parts = []
    parts.append(f"Product: {row['title']}.")
    parts.append(f"Category: {row['category_name']}.")
    if row['price'] > 0:
        parts.append(f"Price: ${row['price']:.2f}.")
    if row['listPrice'] > 0 and row['price'] > 0:
        discount = ((row['listPrice'] - row['price']) / row['listPrice']) * 100
        if discount > 0:
            parts.append(f"Original price: ${row['listPrice']:.2f} (discount: {discount:.0f}%).")
    if row['stars'] > 0:
        parts.append(f"Rating: {row['stars']} out of 5.")
    if row['reviews'] > 0:
        parts.append(f"Number of reviews: {row['reviews']}.")
    if row['isBestSeller']:
        parts.append("This product is a Best Seller.")
    if row['boughtInLastMonth'] > 0:
        parts.append(f"Bought {row['boughtInLastMonth']} times in the last month.")
    return ' '.join(parts)

# show a few sample contexts
sample_rows = df.sample(5, random_state=42)
for idx, row in sample_rows.iterrows():
    print(build_context(row))
    print('---')

In [ ]:
# examine the types of questions that can be answered from this dataset
question_types = [
    ("Price query", "What is the price of [product]?"),
    ("Comparison", "Which product is cheaper, [A] or [B]?"),
    ("Rating query", "What is the rating of [product]?"),
    ("Category query", "What category does [product] belong to?"),
    ("Recommendation", "What is the best rated product in [category]?"),
    ("Discount query", "Is [product] on sale?"),
    ("Popularity query", "How popular is [product]?"),
    ("Best seller", "Is [product] a best seller?"),
]

print('Types of questions the QA system should handle:')
for qtype, example in question_types:
    print(f'  {qtype}: "{example}"')

## 11. Summary of Findings and Limitations

### Key Findings
- The dataset contains approximately 100,000 Amazon products across ~250 categories.
- There is a strong class imbalance across categories, with some categories having thousands of products while others have very few.
- A significant number of products have zero price, zero rating, or zero reviews, which limits the information available for generating answers.
- Product titles are the primary textual feature and vary widely in length and informativeness.
- Price distributions are heavily right-skewed, with most products priced under $50.
- Very few products are marked as Best Sellers.

### Limitations
- **No product descriptions**: The dataset only contains product titles, not full descriptions. This limits the amount of contextual text available for a QA system.
- **No actual Q&A pairs**: Unlike standard QA datasets (SQuAD, Natural Questions), we do not have real question-answer pairs. We will need to generate synthetic QA pairs from the structured metadata.
- **Missing data**: Many products lack reviews, ratings, or price information, which reduces the answerable scope.
- **Category imbalance**: The uneven distribution across categories means the model may perform better on well-represented categories.
- **No user reviews text**: Only review counts are available, not the actual review text, which would have been valuable for a QA system.